# 13 · AWS compacto — indicador v3

Ejecutar después de 09 COMPLETADA. Usa **psycopg2 y SSL** y las credenciales de `.env.aws`. No instala dependencias, no recalcula el modelo y no sube todos los campos de Gold.

Ahora hay **11 columnas** en `analytics_v3.indicadores`: ID, C1/C2/C3/V1/V2/P1/P2, puntaje, clasificación y alerta binaria. Sigue siendo una proyección compacta. Los motivos y métricas detalladas quedan localmente para Grafana/EDA. Corte, parámetros y huellas se guardan una sola vez en `analytics_v3.publicacion`.

Los agregados globales se guardan en `analytics_v3.clasificaciones` y `analytics_v3.dimensiones`. **Las tablas AWS v2 no se sobrescriben.** Dentro de v3 se mantiene una fotografía vigente, sustituida atómicamente; el historial completo se conserva en los lotes locales.


In [1]:
from pathlib import Path
import sys,json
from IPython.display import display,Markdown
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'secop_pipeline').is_dir()),None)
if ROOT is None:raise RuntimeError('Abra el notebook dentro del proyecto.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from secop_pipeline.indicador_v3 import latest_result,COMPACT
from secop_pipeline.indicador_v3_publicacion import inspect_result,publish,prove
from secop_pipeline.alertas_aws import check_connection,rds_engine
RESULTADOS=latest_result(ROOT)
_,reporte,_=inspect_result(RESULTADOS)
PUBLICAR=True
display(check_connection())
display({'columnas_compactas':COMPACT,'filas':reporte['resumen']['contratos'],'version':reporte['version']})


{'conexion': 'verificada', 'ssl': True}

{'columnas_compactas': ['id_contrato',
  'c1',
  'c2',
  'c3',
  'v1',
  'v2',
  'p1',
  'p2',
  'puntaje',
  'clasificacion',
  'alerta_binaria'],
 'filas': 4353720,
 'version': 'alertas-cd-3.0'}

## Publicar y consultar

COPY envía solo las 11 columnas en lotes de 50.000 filas. Se validan conteos y agregados antes del COMMIT. El porcentaje de envío no incluye el tiempo posterior de integración. No evaluable conserva puntaje/alerta NULL. Repetir un archivo ya publicado omite COPY si su huella y conteo coinciden.


In [2]:
if PUBLICAR:
    resultado_aws=publish(RESULTADOS)
    display(resultado_aws)
else:print('Publicación omitida.')
import pandas as pd
from sqlalchemy import text
engine=rds_engine()
try:
    with engine.connect() as conn:
        conn.execute(text('SET TRANSACTION READ ONLY'))
        for tabla in ('clasificaciones','dimensiones'):
            print(tabla)
            display(pd.read_sql_query(text(f'SELECT * FROM analytics_v3.{tabla}'),conn))
finally:engine.dispose()


AWS v3: 50,000/4,353,720 (1.1%); 2.3 MiB
AWS v3: 100,000/4,353,720 (2.3%); 4.5 MiB
AWS v3: 150,000/4,353,720 (3.4%); 6.8 MiB
AWS v3: 200,000/4,353,720 (4.6%); 9.0 MiB
AWS v3: 250,000/4,353,720 (5.7%); 11.3 MiB
AWS v3: 300,000/4,353,720 (6.9%); 13.5 MiB
AWS v3: 350,000/4,353,720 (8.0%); 15.8 MiB
AWS v3: 400,000/4,353,720 (9.2%); 18.1 MiB
AWS v3: 450,000/4,353,720 (10.3%); 20.3 MiB
AWS v3: 500,000/4,353,720 (11.5%); 22.6 MiB
AWS v3: 550,000/4,353,720 (12.6%); 24.9 MiB
AWS v3: 600,000/4,353,720 (13.8%); 27.1 MiB
AWS v3: 650,000/4,353,720 (14.9%); 29.4 MiB
AWS v3: 700,000/4,353,720 (16.1%); 31.6 MiB
AWS v3: 750,000/4,353,720 (17.2%); 33.9 MiB
AWS v3: 800,000/4,353,720 (18.4%); 36.2 MiB
AWS v3: 850,000/4,353,720 (19.5%); 38.4 MiB
AWS v3: 900,000/4,353,720 (20.7%); 40.7 MiB
AWS v3: 950,000/4,353,720 (21.8%); 43.0 MiB
AWS v3: 1,000,000/4,353,720 (23.0%); 45.2 MiB
AWS v3: 1,050,000/4,353,720 (24.1%); 47.5 MiB
AWS v3: 1,100,000/4,353,720 (25.3%); 49.7 MiB
AWS v3: 1,150,000/4,353,720 (26.4%); 52

{'estado': 'COMMIT',
 'bytes_csv': 206468369,
 'filas': 4353720,
 'clasificaciones': {'ALERTA': 42587,
  'NO_ALERTA': 186084,
  'NO_EVALUABLE': 4125049},
 'dimensiones_verificadas': 7}

clasificaciones


,clasificacion,contratos,porcentaje
0,ALERTA,42587,0.978175
1,NO_ALERTA,186084,4.274138
2,NO_EVALUABLE,4125049,94.747687


dimensiones


,dimension,total,evaluados,alertas,sin_senal,no_evaluables,porcentaje_alertas_evaluados,porcentaje_cobertura
0,C1,4353720,4330105,206,4329899,23615,0.004757,99.457590
1,C2,4353720,4343765,369,4343396,9955,0.008495,99.771345
2,C3,4353720,262261,69082,193179,4091459,26.340935,6.023837
3,V1,4353720,4246400,13340,4233060,107320,0.314148,97.534982
4,V2,4353720,4332972,13057,4319915,20748,0.301341,99.523442
5,P1,4353720,4349673,1024545,3325128,4047,23.554529,99.907045
6,P2,4353720,248202,57264,190938,4105518,23.071530,5.700918


## Prueba real de AWS — independiente de la carga

Puedes ejecutar solo la siguiente celda. Abre una conexión nueva de solo lectura y compara conteos, columnas, huella, clasificaciones y las siete dimensiones con el lote local. La prueba antigua demostraba la carga **v2**; no demuestra que v3 esté publicado. Aquí no hay resultados prellenados: aparecerán después de tu ejecución.


In [3]:
from pathlib import Path
import sys,json
from IPython.display import display,Markdown
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'secop_pipeline').is_dir()),None)
if ROOT is None:raise RuntimeError('Abra el notebook dentro del proyecto.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from secop_pipeline.indicador_v3 import latest_result
from secop_pipeline.indicador_v3_publicacion import prove
LOTE_PRUEBA=globals().get('RESULTADOS',None) or latest_result(ROOT)
evidencia=prove(LOTE_PRUEBA)
print(json.dumps(evidencia,ensure_ascii=False,indent=2,default=str))


{
  "estado": "PRUEBA APROBADA",
  "conexion": {
    "base": "basedatosproyectobd",
    "verificado_en": "2026-10-23 11:41:31.409297+00:00",
    "ssl": true
  },
  "columnas": [
    "id_contrato",
    "c1",
    "c2",
    "c3",
    "v1",
    "v2",
    "p1",
    "p2",
    "puntaje",
    "clasificacion",
    "alerta_binaria"
  ],
  "filas": 4353720,
  "clasificaciones": {
    "ALERTA": 42587,
    "NO_ALERTA": 186084,
    "NO_EVALUABLE": 4125049
  },
  "dimensiones_verificadas": 7
}
